In [1]:
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(Path("configs") / "local.env")

#import numpy as np
#from datasets import load_dataset
import torch
torch.manual_seed(3647)
torch.set_float32_matmul_precision('high')

from transformers import set_seed
set_seed(42)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_name = "Qwen/Qwen3-8B"

print(f"--> device: {device}")

--> device: cuda


In [3]:
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

#quant_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16)
#quant_config = None

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    quantization_config=quant_config,
)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [5]:
def chat(messages, max_new_tokens=2048):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True, # Switches between thinking and non-thinking modes. Default is True.
        temperature=0.2,
    )

    tokens = tokenizer([text], return_tensors="pt").to(model.device)

    # conduct text completion
    #generated_ids = model.generate(**tokens, max_new_tokens=32768)
    generated_ids = model.generate(**tokens, max_new_tokens=2048)
    output_ids = generated_ids[0][len(tokens.input_ids[0]):].tolist() 

    # index = len(output_ids) - output_ids[::-1].index(151668) # </think>
    # output_ids[:index]
    content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
    return content

In [6]:
msgs = [
    {"role": "user", "content": "Give me a short introduction to large language model."},
]

print(chat(msgs))

<think>
Okay, the user wants a short introduction to large language models. Let me start by defining what they are. They're AI systems trained on vast amounts of text data, right? So I should mention their size, like the number of parameters. Then explain how they work—like using deep learning and neural networks. Maybe mention their ability to understand and generate human-like text. Also, include applications such as answering questions, writing, coding, etc. Need to keep it concise but cover the key points. Avoid jargon so it's accessible. Let me check if I'm missing anything important, like the training process or specific examples. Oh, and maybe touch on their impact or significance in AI. Alright, time to put it all together in a clear, brief manner.
</think>

Large language models (LLMs) are advanced AI systems trained on massive datasets of text, enabling them to understand, generate, and respond to human language with remarkable fluency. Built using deep learning techniques, t

In [7]:
msgs = [
    {"role": "user", "content": "简要的介绍一下大语言模型(large language models)。"},
]

print(chat(msgs))

<think>
嗯，用户让我简要介绍一下大语言模型。首先，我需要确定他们对这个主题的了解程度。可能他们刚开始接触AI，或者只是想快速了解基本概念。所以得保持简洁，避免太技术性的术语。

接下来，我应该从定义开始，说明大语言模型是什么，比如基于深度学习的模型，处理大量文本数据。然后要提到它们的核心能力，比如自然语言处理，生成文本，回答问题，翻译等。还要区分传统模型和大语言模型的区别，比如参数量更大，训练数据更多。

用户可能想知道这些模型的应用场景，比如聊天机器人、内容创作、编程帮助等。需要举几个例子，比如GPT、BERT，这样更具体。同时，要提到它们的优缺点，比如强大的生成能力，但也可能有幻觉或偏见的问题。

还要考虑用户可能的深层需求。他们可能想了解这些模型在实际中的使用，或者是否适合自己使用。所以需要涵盖应用场景和挑战，比如计算资源需求高，数据隐私问题等。

最后，确保结构清晰，分点说明，语言通俗易懂，避免冗长。检查有没有遗漏的重要点，比如预训练和微调的区别，或者模型的规模如何影响性能。保持回答简明，但覆盖关键方面。
</think>

大语言模型（Large Language Models, LLMs）是基于深度学习技术的自然语言处理模型，通过海量文本数据训练，具备强大的语言理解和生成能力。其核心特点包括：

1. **规模与能力**  
   - 拥有数十亿至万亿级参数，通过预训练（如语言建模）和微调（如任务特定训练）实现。
   - 能够理解上下文、生成连贯文本、回答问题、翻译、编程等，甚至进行逻辑推理和创造性写作。

2. **核心技术**  
   - 基于Transformer架构，通过自注意力机制捕捉长距离依赖关系。
   - 依赖大规模语料库（如书籍、网页、对话）进行无监督学习。

3. **应用场景**  
   - 聊天机器人（如ChatGPT）、内容创作、代码生成、多语言翻译、教育辅导等。
   - 支持个性化服务（如定制化对话、知识问答）。

4. **挑战与局限**  
   - 可能产生“幻觉”（虚构信息）、偏见或敏感内容。
   - 对计算资源要求高，存在数据隐私和伦理风险。

**典型代表**：GPT系列（OpenAI）、BERT（Google）、PaLM（Google）、Llama（Meta）等。  
当前研究方向包括模型压缩、